# A3. AI-Assisted Advertising Optimization: Search Term Report Analysis

> Companion module: [A3 Advertising Optimization](../paths/a-operators/a3-advertising.md)
>
> This Notebook demonstrates how to use Python + AI to analyze Amazon search term reports, identifying high-ROAS terms, wasted spend, and hidden opportunities.
>
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kangise/ecommerce-ai-roadmap/blob/main/notebooks/a3-advertising.ipynb)

## 1. Setup

This Notebook does not require an API Key — it uses pure Python for data processing.
The AI analysis section provides Prompt templates that you can copy into ChatGPT/Claude.

In [ ]:
import pandas as pd
import numpy as np

print('Environment ready ✅')

## 2. Simulated Search Term Report Data

In practice, download the CSV from Amazon Seller Central → Advertising → Search Term Report.
Here we use simulated data to demonstrate the analysis workflow.

In [ ]:
# Simulated Amazon search term report
np.random.seed(42)

search_terms = [
    'portable neck fan', 'neck fan usb', 'hands free fan', 'personal fan',
    'neck fan rechargeable', 'bladeless neck fan', 'cooling fan portable',
    'fan for running', 'outdoor neck fan', 'best neck fan 2026',
    'mini fan', 'desk fan usb', 'clip on fan', 'stroller fan',
    'neck fan for kids', 'neck fan quiet', 'neck fan long battery',
    'wearable fan', 'travel fan portable', 'neck air conditioner',
    'cheap fan', 'fan amazon', 'cooling device', 'summer gadgets',
    'neck fan pink', 'neck fan black', 'fan with led', 'sports fan neck',
    'construction fan', 'warehouse fan'
]

data = {
    'Search Term': search_terms,
    'Impressions': np.random.randint(500, 50000, len(search_terms)),
    'Clicks': np.random.randint(5, 500, len(search_terms)),
    'Spend': np.round(np.random.uniform(1, 200, len(search_terms)), 2),
    'Orders': np.random.randint(0, 30, len(search_terms)),
    'Sales': np.round(np.random.uniform(0, 600, len(search_terms)), 2)
}

df = pd.DataFrame(data)
# Ensure logical consistency
df.loc[df['Orders'] == 0, 'Sales'] = 0
df['CTR'] = np.round(df['Clicks'] / df['Impressions'] * 100, 2)
df['CPC'] = np.round(df['Spend'] / df['Clicks'], 2)
df['ACOS'] = np.where(df['Sales'] > 0, np.round(df['Spend'] / df['Sales'] * 100, 1), np.inf)
df['ROAS'] = np.where(df['Spend'] > 0, np.round(df['Sales'] / df['Spend'], 2), 0)
df['CVR'] = np.where(df['Clicks'] > 0, np.round(df['Orders'] / df['Clicks'] * 100, 2), 0)

print(f'Number of search terms: {len(df)}')
print(f'Total spend: ${df["Spend"].sum():.2f}')
print(f'Total sales: ${df["Sales"].sum():.2f}')
print(f'Overall ROAS: {df["Sales"].sum() / df["Spend"].sum():.2f}x')
df.head(10)

## 3. Search Term Classification: Four-Quadrant Analysis

Classify search terms by ROAS and spend into four categories:
- 🟢 **Star Terms**: High ROAS + High spend → Increase investment
- 🔵 **Potential Terms**: High ROAS + Low spend → Raise bids to capture more traffic
- 🟡 **Watch Terms**: Low ROAS + Low spend → Continue monitoring or optimize
- 🔴 **Wasted Terms**: Low ROAS + High spend → Lower bids or negate

In [ ]:
# Set thresholds
ROAS_THRESHOLD = 3.0  # ROAS >= 3 is considered high
SPEND_THRESHOLD = df['Spend'].median()  # Median spend as the dividing line

def classify_term(row):
    if row['Orders'] == 0 and row['Spend'] > 10:
        return '🔴 Wasted (Zero Conversion)'
    elif row['ROAS'] >= ROAS_THRESHOLD and row['Spend'] >= SPEND_THRESHOLD:
        return '🟢 Star Term'
    elif row['ROAS'] >= ROAS_THRESHOLD and row['Spend'] < SPEND_THRESHOLD:
        return '🔵 Potential Term'
    elif row['ROAS'] < ROAS_THRESHOLD and row['Spend'] >= SPEND_THRESHOLD:
        return '🔴 Wasted Term'
    else:
        return '🟡 Watch Term'

df['Category'] = df.apply(classify_term, axis=1)

print('=== Search Term Classification Summary ===')
print(df['Category'].value_counts())
print()

for cat in ['🟢 Star Term', '🔵 Potential Term', '🔴 Wasted Term', '🔴 Wasted (Zero Conversion)', '🟡 Watch Term']:
    subset = df[df['Category'] == cat]
    if len(subset) > 0:
        print(f'\n--- {cat} ({len(subset)} terms) ---')
        print(subset[['Search Term', 'Spend', 'Sales', 'ROAS', 'Orders']].to_string(index=False))

## 4. Negative Keyword Candidates

Automatically identify search terms that should be negated: spend > $10 with zero conversions, or ROAS < 1.

In [ ]:
# Negative keyword candidates
negative_candidates = df[
    ((df['Orders'] == 0) & (df['Spend'] > 10)) |
    ((df['ROAS'] < 1) & (df['ROAS'] > 0) & (df['Spend'] > 20))
].sort_values('Spend', ascending=False)

print(f'=== Recommended Negative Keywords ({len(negative_candidates)} terms) ===')
print(f'Negating these terms can save: ${negative_candidates["Spend"].sum():.2f}')
print()
print(negative_candidates[['Search Term', 'Spend', 'Orders', 'ROAS']].to_string(index=False))

## 5. Bid Optimization Suggestions

Provide bid adjustment recommendations for each search term based on ROAS.

In [ ]:
def bid_suggestion(row):
    if row['Orders'] == 0:
        return 'Negate or pause'
    elif row['ROAS'] >= 5:
        return f'Increase bid +20% (current CPC ${row["CPC"]:.2f})'
    elif row['ROAS'] >= 3:
        return f'Maintain current bid (CPC ${row["CPC"]:.2f})'
    elif row['ROAS'] >= 1.5:
        return f'Lower bid -15% (current CPC ${row["CPC"]:.2f})'
    else:
        return f'Lower bid -30% or negate (current CPC ${row["CPC"]:.2f})'

df['Bid Suggestion'] = df.apply(bid_suggestion, axis=1)

print('=== Bid Optimization Suggestions ===')
for _, row in df.sort_values('Spend', ascending=False).head(15).iterrows():
    print(f'{row["Search Term"]:30s} | ROAS {row["ROAS"]:5.2f}x | {row["Bid Suggestion"]}')

## 6. Generate AI Analysis Prompt

Compile the data into a Prompt — copy it into ChatGPT/Claude for deeper analysis.

In [ ]:
# Generate AI Prompt
top_terms = df.nlargest(20, 'Spend')[['Search Term', 'Impressions', 'Clicks', 'Spend', 'Orders', 'Sales', 'ROAS', 'ACOS']]

prompt = f"""You are an Amazon PPC advertising optimization expert.

Below is my search term report data (past 14 days, Top 20 by spend):

Total Spend: ${df['Spend'].sum():.2f}
Total Sales: ${df['Sales'].sum():.2f}
Overall ROAS: {df['Sales'].sum() / df['Spend'].sum():.2f}x

{top_terms.to_markdown(index=False)}

Please analyze:
1. Which are high-value terms? How much should bids be increased?
2. Which terms are wasting budget? Should they be negated or have bids lowered?
3. Are there hidden long-tail opportunities?
4. Overall advertising strategy recommendations (budget allocation, new keyword directions)
"""

print('=== Copy the following Prompt into ChatGPT/Claude ===')
print(prompt)

## 7. Summary

This Notebook demonstrated:
1. Four-quadrant search term classification (Star / Potential / Wasted / Watch terms)
2. Automatic negative keyword candidate generation
3. ROAS-based bid optimization suggestions
4. Automatic AI analysis Prompt generation

**Next steps**:
- Replace the simulated data with your own search term report CSV
- Copy the generated Prompt into ChatGPT/Claude for deeper analysis
- Refer to [A3 Advertising Optimization](../paths/a-operators/a3-advertising.md) for the complete advertising optimization methodology

---
📎 **Source**: [ecommerce-ai-roadmap](https://github.com/kangise/ecommerce-ai-roadmap)